In [ ]:
# ============================================================
# Dataset : Student AI Usage Survey
# Source: archive (5).zip -> Students.csv
# Used by: Dashboard 5 (Student AI Usage Analytics), Dashboard 4 (AI Dependency)
#
# Raw data is mostly clean (no duplicates), but has inconsistent
# text casing (e.g. "Uttar pradesh") and 1614/3614 missing State
# values. Missing State is filled as "Unknown" rather than dropped,
# since dropping would remove 45% of rows and bias the regional
# breakdown in the dashboards.
#
# Cleaning steps:
# 1. Strip whitespace from all text columns
# 2. Title-case free-text categorical columns (State, Stream, College_Name)
# 3. Fill missing State as "Unknown"
# 4. Convert Yes/No columns to real booleans
# 5. Split multi-value AI_Tools_Used into a list + Primary_AI_Tool
# 6. Validate/clip numeric fields to their expected ranges
#    (Daily_Usage_Hours, Trust_in_AI_Tools, Impact_on_Grades, Awareness_Level)
# 7. Derive Academic_Use_Flag and Usage_Bucket for the dependency dashboard
# ============================================================

In [25]:
import zipfile

zip_path = "/Users/ananyadubey/Downloads/archive (5).zip"
extract_to = "/Users/ananyadubey/Downloads/student_survey"

with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(extract_to)

import os
print(os.listdir(extract_to))   # should show ['Students.csv']

['Students.csv']


In [26]:
import pandas as pd
import numpy as np

df = pd.read_csv("/Users/ananyadubey/Downloads/student_survey/Students.csv")
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())

(3614, 16)
Student_Name                  object
College_Name                  object
Stream                        object
Year_of_Study                  int64
AI_Tools_Used                 object
Daily_Usage_Hours            float64
Use_Cases                     object
Trust_in_AI_Tools              int64
Impact_on_Grades               int64
Do_Professors_Allow_Use       object
Preferred_AI_Tool             object
Awareness_Level                int64
Willing_to_Pay_for_Access     object
State                         object
Device_Used                   object
Internet_Access               object
dtype: object
Student_Name                    0
College_Name                    0
Stream                          0
Year_of_Study                   0
AI_Tools_Used                   0
Daily_Usage_Hours               0
Use_Cases                       0
Trust_in_AI_Tools               0
Impact_on_Grades                0
Do_Professors_Allow_Use         0
Preferred_AI_Tool               0
Awareness

In [27]:
str_cols = df.select_dtypes(include="object").columns
for c in str_cols:
    df[c] = df[c].astype(str).str.strip()

In [28]:
#Fix inconsistent casing
for c in ["State", "Stream", "College_Name"]:
    df[c] = df[c].replace("nan", np.nan)   # str.strip() turned real NaNs into the string "nan"
    df[c] = df[c].str.title()

In [29]:
#Decide what to do with missing State
df["State"] = df["State"].fillna("Unknown")

In [30]:
#Convert Yes/No text to real booleans
for c in ["Do_Professors_Allow_Use", "Willing_to_Pay_for_Access"]:
    df[c] = df[c].str.title().map({"Yes": True, "No": False})

In [31]:
#Validate numeric ranges
range_checks = {
    "Daily_Usage_Hours": (0, 24),
    "Trust_in_AI_Tools": (1, 5),
    "Impact_on_Grades": (-5, 5),
    "Awareness_Level": (1, 10),
}
for col, (lo, hi) in range_checks.items():
    bad = ~df[col].between(lo, hi)
    print(col, "out-of-range rows:", bad.sum())
    df[col] = df[col].clip(lo, hi)

Daily_Usage_Hours out-of-range rows: 0
Trust_in_AI_Tools out-of-range rows: 0
Impact_on_Grades out-of-range rows: 0
Awareness_Level out-of-range rows: 0


In [32]:
#  split multi-tool field
df["AI_Tools_Used_List"] = df["AI_Tools_Used"].apply(lambda x: [t.strip() for t in str(x).split(",")])
df["Primary_AI_Tool"] = df["AI_Tools_Used_List"].apply(lambda x: x[0])
df["Num_AI_Tools_Used"] = df["AI_Tools_Used_List"].apply(len)

# derived fields for dependency dashboard
df["Academic_Use_Flag"] = df["Use_Cases"].str.contains("Assignment|Research|Learning", case=False, na=False)
df["Usage_Bucket"] = pd.cut(df["Daily_Usage_Hours"], bins=[-0.01, 1, 3, 24], labels=["Low (<1h)", "Medium (1-3h)", "High (3h+)"])

# drop the list column and save
export_df = df.drop(columns=["AI_Tools_Used_List"])
print(export_df.columns.tolist())   # should now show 20 columns
export_df.to_csv("/Users/ananyadubey/student_survey_clean.csv", index=False)

['Student_Name', 'College_Name', 'Stream', 'Year_of_Study', 'AI_Tools_Used', 'Daily_Usage_Hours', 'Use_Cases', 'Trust_in_AI_Tools', 'Impact_on_Grades', 'Do_Professors_Allow_Use', 'Preferred_AI_Tool', 'Awareness_Level', 'Willing_to_Pay_for_Access', 'State', 'Device_Used', 'Internet_Access', 'Primary_AI_Tool', 'Num_AI_Tools_Used', 'Academic_Use_Flag', 'Usage_Bucket']


In [33]:
#verify
df_check = pd.read_csv("/Users/ananyadubey/student_survey_clean.csv")
print(df_check.shape)
print(df_check['State'].unique()[:10])
print(df_check['Do_Professors_Allow_Use'].dtype)
print(df_check[['Daily_Usage_Hours','Trust_in_AI_Tools']].describe())

(3614, 20)
['Uttar Pradesh' 'Chhattisgarh' 'Uttarakhand' 'Delhi Ncr' 'Punjab'
 'Chandigarh' 'Puducherry' 'Rajasthan' 'Meghalaya' 'Jharkhand']
bool
       Daily_Usage_Hours  Trust_in_AI_Tools
count        3614.000000        3614.000000
mean            2.559685           3.023243
std             1.213319           1.436934
min             0.500000           1.000000
25%             1.500000           2.000000
50%             2.600000           3.000000
75%             3.600000           4.000000
max             5.000000           5.000000


In [34]:
df_final = pd.read_csv("/Users/ananyadubey/student_survey_clean.csv")
print(df_final.shape)
print(df_final[['Primary_AI_Tool','Num_AI_Tools_Used','Academic_Use_Flag','Usage_Bucket']].head())
print(df_final['Usage_Bucket'].value_counts())

(3614, 20)
  Primary_AI_Tool  Num_AI_Tools_Used  Academic_Use_Flag   Usage_Bucket
0          Gemini                  1               True      Low (<1h)
1         ChatGPT                  1               True     High (3h+)
2         Copilot                  1              False     High (3h+)
3         Copilot                  1              False  Medium (1-3h)
4          Gemini                  1              False      Low (<1h)
Usage_Bucket
Medium (1-3h)    1658
High (3h+)       1442
Low (<1h)         514
Name: count, dtype: int64


In [ ]:
# ============================================================
# Dataset : ChatGPT Prompts Dataset (Awesome ChatGPT Prompts)
# Source: archive (3).zip -> prompts.csv
# Used by: Dashboard 3 (Prompt Analytics)
#
# Note: this is NOT real user logs — it's 153 curated PROMPT
# TEMPLATES (role-play style "act as X" prompts). Useful for
# category/keyword reference, but won't give volume metrics
# the way real usage data would. Dashboard 3's "Prompt Length
# Distribution" etc. should be read as template-based here,
# not user-log-based.
#
# Cleaning steps:
# 1. Strip whitespace, normalize internal newlines/spacing
# 2. Drop exact duplicate prompts
# 3. Add Prompt_Length (word count)
# 4. Add Prompt_Category via rule-based keyword matching,
#    aligned to the 6 categories used in Dashboard 3
# ============================================================

In [36]:
import zipfile
import os

zip_path = "/Users/ananyadubey/Downloads/archive (3).zip"
extract_to = "/Users/ananyadubey/Downloads/chatgpt_prompts"

with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(extract_to)

print(os.listdir(extract_to))   # should show ['prompts.csv']

['prompts.csv']


In [37]:
import pandas as pd

df2 = pd.read_csv("/Users/ananyadubey/Downloads/chatgpt_prompts/prompts.csv")
print(df2.shape)
print(df2.columns.tolist())
print(df2.head(3))

(153, 2)
['act', 'prompt']
                               act  \
0                   Linux Terminal   
1  English Translator and Improver   
2           `position` Interviewer   

                                              prompt  
0  I want you to act as a linux terminal. I will ...  
1  I want you to act as an English translator, sp...  
2  I want you to act as an interviewer. I will be...  


In [38]:
import re

for c in ["act", "prompt"]:
    df2[c] = df2[c].astype(str).str.strip()
    df2[c] = df2[c].apply(lambda x: re.sub(r"\s+", " ", x))

In [39]:
print("Duplicate prompts:", df2.duplicated(subset=["prompt"]).sum())
df2 = df2.drop_duplicates(subset=["prompt"])

Duplicate prompts: 0


In [40]:
df2["Prompt_Length"] = df2["prompt"].apply(lambda x: len(x.split()))

In [41]:
CATEGORY_KEYWORDS = {
    "Coding": ["code", "program", "developer", "javascript", "python", "sql", "linux", "terminal", "debug"],
    "Learning": ["teach", "explain", "tutor", "learn", "lesson"],
    "Writing": ["write", "essay", "story", "novelist", "poet", "translator", "editor"],
    "Research": ["research", "analy", "data", "summariz"],
    "Decision Making": ["advisor", "consultant", "advice", "recommend", "decide"],
    "Entertainment": ["game", "joke", "fun", "role play", "character", "magician"],
}

def categorize(text, act):
    combined = f"{act} {text}".lower()
    for cat, keywords in CATEGORY_KEYWORDS.items():
        if any(k in combined for k in keywords):
            return cat
    return "Other"

df2["Prompt_Category"] = df2.apply(lambda r: categorize(r["prompt"], r["act"]), axis=1)

In [42]:
print(df2["Prompt_Category"].value_counts())

Prompt_Category
Writing            44
Other              32
Coding             26
Learning           19
Research           13
Decision Making    12
Entertainment       7
Name: count, dtype: int64


In [43]:
df2.to_csv("/Users/ananyadubey/chatgpt_prompts_clean.csv", index=False)
print(df2.shape)

(153, 4)


In [ ]:
# ============================================================
# Dataset : OpenAssistant (OASST1)
# Source: archive (2).zip -> oasst1_train.csv, oasst1_validation.csv
# Used by: Dashboard 3 (Prompt Analytics), Dashboard 4 (AI Dependency)
# Note: this is a conversation TREE not a flat table — message_tree_id
# groups messages in one thread, role = prompter (user) or assistant
# ============================================================

In [44]:
# unzip the dataset
import zipfile, os

zip_path = "/Users/ananyadubey/Downloads/archive (2).zip"
extract_to = "/Users/ananyadubey/Downloads/oasst"

with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(extract_to)

print(os.listdir(extract_to))

['oasst1_validation.csv', 'oasst1_train.csv']


In [45]:
# only load the columns we actually need, file has heavy nested
# columns (detoxify, emojis, labels) that'll eat memory for nothing
USECOLS = [
    "message_id", "parent_id", "user_id", "created_date", "text",
    "role", "lang", "deleted", "rank", "message_tree_id", "tree_state",
]

train = pd.read_csv("/Users/ananyadubey/Downloads/oasst/oasst1_train.csv", usecols=USECOLS)
val = pd.read_csv("/Users/ananyadubey/Downloads/oasst/oasst1_validation.csv", usecols=USECOLS)

In [46]:
# tag which split each row came from before merging, so we can
# trace it back later if needed
train["split"] = "train"
val["split"] = "validation"

df3 = pd.concat([train, val], ignore_index=True)
print(df3.shape)
print(df3.columns.tolist())
print(df3.head(3))

(88838, 12)
['message_id', 'parent_id', 'user_id', 'created_date', 'text', 'role', 'lang', 'deleted', 'rank', 'message_tree_id', 'tree_state', 'split']
                             message_id                             parent_id  \
0  6ab24d72-0181-4594-a9cd-deaf170242fb                                   NaN   
1  c8e83833-ecbc-44fe-b6db-735228c25a1c  6ab24d72-0181-4594-a9cd-deaf170242fb   
2  6708c47f-05c9-4346-b3d2-40b2bd24fde4  c8e83833-ecbc-44fe-b6db-735228c25a1c   

                                user_id                      created_date  \
0  c3fe8c76-fc30-4fa7-b7f8-c492f5967d18  2023-02-05T14:23:50.983374+00:00   
1  2c96e467-66f0-4be7-9693-bda51356a424  2023-02-06T13:50:44.657083+00:00   
2  2c96e467-66f0-4be7-9693-bda51356a424  2023-02-06T18:48:49.391686+00:00   

                                                text       role lang  deleted  \
0  Can you write a short introduction about the r...   prompter   en    False   
1  "Monopsony" refers to a market structure where...

In [47]:
# drop deleted messages — they shouldn't count toward any analytics
df3 = df3[df3["deleted"] != True]
print(df3.shape)

(87285, 12)


In [48]:
# drop rows with missing text just in case
df3 = df3.dropna(subset=["text"])
print(df3.shape)

(87285, 12)


In [49]:
# filter to english only — dataset has many languages, keeping it
# simple for now since most dashboards expect english text
before = len(df3)
df3 = df3[df3["lang"] == "en"]
print(f"{before} -> {len(df3)} rows after english filter")

87285 -> 40436 rows after english filter


In [50]:
# parse timestamp properly instead of leaving it as text
df3["created_date"] = pd.to_datetime(df3["created_date"], errors="coerce")

In [51]:
# clean up extra whitespace/newlines inside message text
import re
df3["text"] = df3["text"].astype(str).apply(lambda x: re.sub(r"\s+", " ", x.strip()))

In [52]:
# filter to english only — dataset has many languages, keeping it
# simple for now since most dashboards expect english text
before = len(df3)
df3 = df3[df3["lang"] == "en"]
print(f"{before} -> {len(df3)} rows after english filter")

40436 -> 40436 rows after english filter


In [53]:
# parse timestamp properly instead of leaving it as text
df3["created_date"] = pd.to_datetime(df3["created_date"], errors="coerce")

In [54]:
# clean up extra whitespace/newlines inside message text
import re
df3["text"] = df3["text"].astype(str).apply(lambda x: re.sub(r"\s+", " ", x.strip()))

In [55]:
# split off just the user messages (role == prompter) — these are
# the actual prompts people typed, separate from assistant replies
prompts = df3[df3["role"] == "prompter"].copy()
print(prompts.shape)

(15708, 12)


In [56]:
# basic length stats per prompt
prompts["Word_Count"] = prompts["text"].str.split().str.len()
prompts["Char_Count"] = prompts["text"].str.len()

In [57]:
# count how many prompts exist in each conversation tree —
# this acts like a "session length" for this dataset
tree_counts = prompts.groupby("message_tree_id").size().rename("Prompts_In_Tree")
prompts = prompts.merge(tree_counts, on="message_tree_id", how="left")

In [58]:
# flag repeated prompts per user — same user asking near-identical
# things more than once, feeds into the redundancy metric
prompts["text_norm"] = prompts["text"].str.lower().str.strip()
dupe_counts = prompts.groupby(["user_id", "text_norm"]).size().rename("Repeat_Count")
prompts = prompts.merge(dupe_counts, on=["user_id", "text_norm"], how="left")
prompts["Is_Repeated_Prompt"] = prompts["Repeat_Count"] > 1
prompts = prompts.drop(columns=["text_norm"])

print(prompts.shape)
print("repeated prompt rate:", round(prompts["Is_Repeated_Prompt"].mean() * 100, 2), "%")

(15708, 17)
repeated prompt rate: 0.11 %


In [59]:
# save the full message table (prompter + assistant) — useful later
# for conversation-length / turn-based metrics
df3.to_csv("/Users/ananyadubey/oasst_messages_clean.csv", index=False)
print(df3.shape)

(40436, 12)


In [60]:
# save the prompts-only table — this is what feeds Dashboard 3
# (prompt analytics) and Dashboard 4 (dependency/redundancy)
prompts.to_csv("/Users/ananyadubey/oasst_prompts_clean.csv", index=False)
print(prompts.shape)

(15708, 17)


In [ ]:
# ============================================================
# Dataset 1: LMSYS Chatbot Arena Conversations
# Source: archive.zip -> chatbot_arena_conversations.json
# Used by: Dashboard 1 (Executive), Dashboard 2 (User Behavior), Dashboard 4 (Dependency)
# Note: this is JSON LINES not a JSON array — each line is one
# "battle" where the same user prompt goes to TWO models. Both
# conversation_a and conversation_b contain the same user turns,
# so we only pull user turns from conversation_a to avoid double-counting
# ============================================================

In [61]:
# unzip the dataset
import zipfile, os

zip_path = "/Users/ananyadubey/Downloads/archive.zip"
extract_to = "/Users/ananyadubey/Downloads/lmsys"

with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(extract_to)

print(os.listdir(extract_to))

['chatbot_arena_conversations.json']


In [62]:
# quick peek at the raw format before parsing — confirms it's
# one JSON object per line, not one big array
with open("/Users/ananyadubey/Downloads/lmsys/chatbot_arena_conversations.json") as f:
    first_line = f.readline()
print(first_line[:500])

{"question_id":"58210e39b3fd4441a2bd4a518bb44c2d","model_a":"chatglm-6b","model_b":"koala-13b","winner":"model_b","judge":"arena_user_973","conversation_a":[{"content":"What is the difference between OpenCL and CUDA?","role":"user"},{"content":"OpenCL and CUDA are two different programming models that are used for parallel computing.OpenCL is a general-purpose\u5e76\u884c\u7f16\u7a0b\u63a5\u53e3 that allows developers to write parallel code that can run on any platform that supportsCL, which inc


In [63]:
# stream the file line by line instead of json.load() — loading the
# whole 113MB as one object would be wasteful, and it's not even
# valid as one JSON array anyway (it's JSON lines)
import json

rows = []
with open("/Users/ananyadubey/Downloads/lmsys/chatbot_arena_conversations.json", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        d = json.loads(line)

        # only pull user turns from conversation_a — conversation_b
        # has the same user turns, just different model replies
        user_turns = [m["content"] for m in d["conversation_a"] if m["role"] == "user"]

        for turn_idx, content in enumerate(user_turns, start=1):
            rows.append({
                "question_id": d.get("question_id"),
                "judge_user_id": d.get("judge"),
                "turn_number": turn_idx,
                "total_turns_in_battle": d.get("turn"),
                "prompt_text": content,
                "language": d.get("language"),
                "tstamp": d.get("tstamp"),
                "model_a": d.get("model_a"),
                "model_b": d.get("model_b"),
                "winner": d.get("winner"),
                "flagged_moderation": d.get("openai_moderation", {}).get("flagged", False),
            })

df4 = pd.DataFrame(rows)
print(df4.shape)
print(df4.head(3))

(39316, 11)
                        question_id   judge_user_id  turn_number  \
0  58210e39b3fd4441a2bd4a518bb44c2d  arena_user_973            1   
1  2564acd09e3942fd97657d05282d4389  arena_user_973            1   
2  90bfd142157948aba01931726c888e7f  arena_user_973            1   

   total_turns_in_battle                                        prompt_text  \
0                      1    What is the difference between OpenCL and CUDA?   
1                      1  Why did my parent not invite me to their wedding?   
2                      1                   Fuji vs. Nikon, which is better?   

  language        tstamp           model_a           model_b   winner  \
0  English  1.682352e+09        chatglm-6b         koala-13b  model_b   
1  English  1.682352e+09  oasst-pythia-12b        alpaca-13b      tie   
2  English  1.682352e+09         koala-13b  oasst-pythia-12b  model_b   

   flagged_moderation  
0               False  
1               False  
2               False  


In [64]:
# drop any blank/empty prompts that might've slipped through
df4 = df4[df4["prompt_text"].astype(str).str.strip() != ""]
print(df4.shape)

(39316, 11)


In [65]:
# convert the unix timestamp into a real datetime, then pull out
# hour/date/weekday — needed for the peak-usage-hours and heatmap charts
df4["datetime"] = pd.to_datetime(df4["tstamp"], unit="s", errors="coerce")
df4["hour_of_day"] = df4["datetime"].dt.hour
df4["date"] = df4["datetime"].dt.date
df4["weekday"] = df4["datetime"].dt.day_name()

In [66]:
# word count for prompt length analytics
df4["Word_Count"] = df4["prompt_text"].str.split().str.len()

In [67]:
# drop rows openai's moderation model flagged — these skew the
# usage/dependency analytics and shouldn't be in a clean report
before = len(df4)
df4 = df4[df4["flagged_moderation"] != True]
print(f"dropped {before - len(df4)} flagged rows")

dropped 0 flagged rows


In [68]:
# count total prompts per user (judge) — feeds "sessions per user"
# and "prompt frequency" charts in Dashboard 2
user_counts = df4.groupby("judge_user_id").size().rename("Total_Prompts_By_User")
df4 = df4.merge(user_counts, on="judge_user_id", how="left")
print(df4.shape)

(39316, 17)


In [69]:
# save the cleaned lmsys prompts table
df4.to_csv("/Users/ananyadubey/lmsys_prompts_clean.csv", index=False)
print(df4.shape)
print(df4.columns.tolist())

(39316, 17)
['question_id', 'judge_user_id', 'turn_number', 'total_turns_in_battle', 'prompt_text', 'language', 'tstamp', 'model_a', 'model_b', 'winner', 'flagged_moderation', 'datetime', 'hour_of_day', 'date', 'weekday', 'Word_Count', 'Total_Prompts_By_User']
